# Project 3: Pass/Fail Classifier
**Задача:** предсказать, сдаст ли студент экзамен (binary classification).
**Темы:** logistic regression, predict_proba, decision boundary, accuracy + confusion matrix.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, ConfusionMatrixDisplay

rng = np.random.default_rng(21)
n = 400
df = pd.DataFrame({
    "часы_учёбы":   rng.uniform(0, 12, n).round(1),
    "ср_балл":      rng.uniform(2.0, 5.0, n).round(2),
})
logit = 0.9*df["часы_учёбы"] + 2.2*df["ср_балл"] - 11
df["сдал"] = (rng.uniform(0, 1, n) < 1/(1+np.exp(-logit))).astype(int)
print(df["сдал"].value_counts())
df.head()

In [ ]:
X = df[["часы_учёбы", "ср_балл"]]
y = df["сдал"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

pipe = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression())]).fit(X_tr, y_tr)
y_pred = pipe.predict(X_te)
print("Accuracy:", accuracy_score(y_te, y_pred).round(3))
print(classification_report(y_te, y_pred, target_names=["провал", "сдал"]))
ConfusionMatrixDisplay(confusion_matrix(y_te, y_pred), display_labels=["провал","сдал"]).plot(cmap='Blues')
plt.show()

In [ ]:
# Decision boundary на плоскости двух признаков
xx, yy = np.meshgrid(np.linspace(0, 12, 200), np.linspace(2, 5, 200))
grid = pd.DataFrame(np.column_stack([xx.ravel(), yy.ravel()]), columns=X.columns)
Z = pipe.predict_proba(grid)[:, 1].reshape(xx.shape)
plt.contourf(xx, yy, Z, levels=20, cmap='RdYlGn', alpha=0.6)
plt.colorbar(label='P(сдал)')
plt.scatter(df["часы_учёбы"], df["ср_балл"], c=df["сдал"], cmap='RdYlGn', edgecolors='k', s=15, linewidths=0.3)
plt.contour(xx, yy, Z, levels=[0.5], colors='blue')   # синяя линия = граница 0.5
plt.xlabel('часы учёбы'); plt.ylabel('средний балл'); plt.title('Карта вероятности сдачи'); plt.show()

# Вероятность для конкретного студента
student = pd.DataFrame([[4.0, 3.5]], columns=X.columns)
print("Студент 4ч/балл 3.5 -> P(сдал) =", pipe.predict_proba(student)[0, 1].round(2))

## Выводы
- Модель даёт не только класс, но и ВЕРОЯТНОСТЬ — это ценнее для решений («P=0.55, рискованно»).
- Синяя линия — decision boundary: компенсация «мало часов» высоким средним баллом видна наглядно.
- Confusion matrix показывает, какого типа ошибки чаще: пропуски или ложные тревоги.

## Задания
1. Снизь порог до 0.3 — как изменится recall класса «провал»?
2. Добавь шумовой признак «номер группы» — изменилось ли качество?
3. Сравни с DecisionTreeClassifier(max_depth=3): чья граница решений адекватнее?